# Preparation

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import networkx as nx
import numpy as np
import re
from adjustText import adjust_text

# Upload files

In [ ]:
root_folder = 'PDFs_May_2025/'

data = pd.read_excel(root_folder + 'download_report_v2.xlsx', sheet_name='Sheet1')
# Display the first few rows of the dataframe
data.head()

In [ ]:
# Keep only rows where either 'Downloaded Filename' or 'Manual Downloaded Filename' is not NaN
data = data[data['Downloaded Filename'].notna() | data['Manual Downloaded Filename'].notna()]

In [ ]:
data.columns

# Data Cleaning

In [ ]:
# Create a new field "Published year" by copying the year from "year"
data['Published year'] = data['Published'].str[:4]

# Convert the "Published year" column to numeric type
data['Published year'] = pd.to_numeric(data['Published year'], errors='coerce').astype('Int64')

# Cleaning topics
data['Cleaned Topics'] = data['Top Topics'].str.replace('-', '').str.replace(' ', '').str.lower()

In [ ]:
data

In [ ]:
# Define the mapping of old values to new values
replacement_dict = {
    'UN': 'United Nations',
    'UN Department of Economic and Social Affairs': 'United Nations',
    'UN Environment': 'United Nations',
    'UN Human Settlements Programme': 'United Nations',
    'UN Women': 'United Nations',
    'UNCDF': 'United Nations',
    'UNCTAD': 'United Nations',
    'UNDP': 'United Nations',
    'UNESCO': 'United Nations',
    'UNICEF': 'United Nations',
    'UNICEF Office of Research - Innocenti': 'United Nations',
    'United Nations': 'United Nations',
    'United Nations CEPAL': 'United Nations',
    'United Nations Department of Economic and Social Affairs': 'United Nations',
    'United Nations Development Programme': 'United Nations',
    'United Nations ESCWA': 'United Nations',
    'United Nations Environment Programme': 'United Nations',
    'United Nations Framework Convention on Climate Change': 'United Nations',
    'United Nations Human Development Report Office': 'United Nations',
    'United Nations Human Settlement Programme': 'United Nations',
    'United Nations Industrial Development Organization': 'United Nations',
    'United Nations Office on Drugs and Crime': 'United Nations',
    'United Nations Population Fund': 'United Nations',
    'Washington, DC': 'World Bank',
    'Washington, DC: World Bank': 'World Bank',
    'World Bank': 'World Bank',
    'World Bank Group': 'World Bank',
    'World Bank, Washington DC': 'World Bank',
    'World Bank, Washington, DC': 'World Bank',
    'Pew Research Center: Internet, Science & Tech': 'Pew Research Center',
    'Overseas Development Institute': 'Overseas Development Institute (ODI)',
    'OECD Publishing': 'OECD',
    'Konrad-Adenauer-Stiftung': 'Konrad Adenauer Stiftung',
    'Atlantic Council': 'Atlantic Council of the United States',
    'Africa CDC': 'Africa Centres for Disease Control and Prevention'
}

# Replace the values in the "Source Name" field
data['Cleaned Source'] = data['Source Name'].replace(replacement_dict).str.replace('-', '').str.replace(' ', '').str.lower()


In [ ]:
# Cleaning document type
data['Cleaned Type'] = data['Policy Document Type'].str.replace('-', '').str.replace(' ', '').str.lower()

# Data Analysis

## Most Relevant Document Types

In [ ]:
data['Cleaned Type'].value_counts()

In [ ]:
# Count the occurrences of each cleaned type
cleaned_type_counts = data['Cleaned Type'].value_counts()

# Plot the data as a pie chart
plt.figure(figsize=(10, 8))
cleaned_type_counts.plot(kind='pie', autopct='%1.1f%%', startangle=140, colormap='tab20')
plt.title('Distribution of Cleaned Types')
plt.legend(cleaned_type_counts.index, loc='upper right', bbox_to_anchor=(1.3, 1))
plt.show()

## Documents per year

In [ ]:
# Count the number of documents per year
documents_per_year = data['Published year'].value_counts().sort_index()

# Reindex to include all years from 2015 to 2024
years = range(2010, 2026)
documents_per_year = documents_per_year.reindex(years, fill_value=0)

# Plot the data as a line graph
plt.figure(figsize=(10, 6))
documents_per_year.plot(kind='line', marker='o')
plt.xlabel('Published Year')
plt.ylabel('Number of Documents')
plt.title('Number of Documents per Year')
plt.grid(True)
plt.show()

In [ ]:
documents_per_year

In [ ]:
# Count the number of publications per year and Source Organisation Type
pubs_per_year_type = data.groupby(['Published year', 'Source Organisation Type']).size().unstack(fill_value=0)

# Plot the stacked bar chart
plt.figure(figsize=(12, 7))
pubs_per_year_type.plot(kind='bar', stacked=True, ax=plt.gca())
plt.xlabel('Published Year')
plt.ylabel('Number of Publications')
plt.title('Publications per Year by Source Organisation Type')
plt.legend(title='Source Organisation Type', loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
pubs_per_year_type

In [ ]:
org_type = "NGO"  # <- change to your exact value
top10 = (
    data.loc[data["Source Organisation Type"].eq(org_type), "Source Name"]
        .value_counts()
        .head(10)
        .rename_axis("Source Name")
        .reset_index(name="Count")
)
top10['Source Name']

In [ ]:
# Get the top 5 sources for each Source Organisation Type
top5_per_type = (
    data.groupby('Source Organisation Type')['Source Name']
    .value_counts()
    .groupby(level=0, group_keys=False)
    .nlargest(5)
    .reset_index(name='Count')
)

# Display the result as a table
top5_per_type

## Most Relevant Topics

In [ ]:
# Split the topics by ';' and flatten the list
all_topics = data['Cleaned Topics'].dropna().str.split(';').explode()

# Count the occurrences of each topic
topic_counts = Counter(all_topics)

# Convert the counter to a DataFrame for better visualization
topic_counts_df = pd.DataFrame.from_dict(topic_counts, orient='index', columns=['Count']).reset_index()
topic_counts_df.rename(columns={'index': 'Topic'}, inplace=True)

# Sort the DataFrame by count in descending order
topic_counts_df = topic_counts_df.sort_values(by='Count', ascending=False)

# Display the top 10 most cited topics
topic_counts_df.head(20)

In [ ]:
'''
#Export topics to excel
file_topic = 'Results/doc_topics.xlsx'
topic_counts_df.to_excel(file_topic, index=False)

In [ ]:
# Plot the top n_topics most relevant topics in a horizontal bar graph
n_topics=10

plt.figure(figsize=(12, 8))
top_topics_sorted = topic_counts_df.head(n_topics).sort_values(by='Count', ascending=True)
top_topics_sorted.plot(kind='barh', x='Topic', y='Count', legend=False)
plt.xlabel('Count')
plt.ylabel('Topic')
plt.title('Top Most Relevant Topics')
plt.xticks(rotation=45)
plt.show()

## Most Relevant Sources

In [ ]:
# Plot the top n_sources most relevant sources in a horizontal bar graph
n_sources = 10

# Count the occurrences of each source
sources_counts = data['Cleaned Source'].value_counts()

# Get the top n_sources
top_sources_sorted = sources_counts.head(n_sources).sort_values(ascending=True)

# Plot the data
plt.figure(figsize=(12, 8))
top_sources_sorted.plot(kind='barh', x='Source', y='Count', legend=False)
plt.xlabel('Count')
plt.ylabel('Source')
plt.title('Top Most Relevant Sources')
plt.xticks(rotation=45)
plt.show()

## Topics per year

In [ ]:
# Filter the data to include only the top 10 topics
top_topics = top_topics_sorted['Topic'].tolist()
filtered_data = data[data['Cleaned Topics'].apply(lambda x: any(topic in x for topic in top_topics) if pd.notna(x) else False)]

# Create a new DataFrame to store the counts per year for each topic
topic_year_counts = pd.DataFrame(index=years, columns=top_topics).fillna(0)

# Count the occurrences of each topic per year
for topic in top_topics:
    topic_data = filtered_data[filtered_data['Cleaned Topics'].str.contains(topic, na=False)]
    topic_counts_per_year = topic_data['Published year'].value_counts().sort_index()
    topic_year_counts[topic] = topic_year_counts.index.map(topic_counts_per_year).fillna(0)

# Plot the data as a line graph
plt.figure(figsize=(14, 8))
for topic in top_topics:
    plt.plot(topic_year_counts.index, topic_year_counts[topic], marker='o', label=topic)

plt.xlabel('Published Year')
plt.ylabel('Number of Documents')
plt.title('Number of Documents per Year for Top Topics')
plt.legend(title='Topics')
plt.grid(True)
plt.show()

In [ ]:
topic_year_counts

# Network of topics

In [ ]:
# Get the top 50 topics based on counts
top_50_topics = topic_counts_df.head(50)['Topic'].tolist()

# Clean and split the 'Top Topics' column
cleaned_topics = data['Cleaned Topics'].dropna().str.split(';')

# Filter the cleaned topics to include only the top 50 topics
filtered_topics = cleaned_topics.apply(lambda x: ';'.join([t for t in x if t in top_50_topics]))

# Create a graph
G = nx.Graph()

# Count the co-occurrences of each pair of topics
co_occurrence_counts = Counter()

for topic in filtered_topics:
    sub_topics = topic.split(';')
    for i in range(len(sub_topics)):
        for j in range(i + 1, len(sub_topics)):
            co_occurrence_counts[(sub_topics[i], sub_topics[j])] += 1

# Add nodes and edges based on the filtered topics
for (topic1, topic2), count in co_occurrence_counts.items():
    G.add_edge(topic1, topic2, weight=count)

# Get the counts for the top 50 topics
node_sizes = [topic_counts_df[topic_counts_df['Topic'] == topic]['Count'].values[0] * 15 for topic in G.nodes()]

# Draw the graph
plt.figure(figsize=(12, 12))
pos = nx.spring_layout(G, k=0.5)
edges = G.edges(data=True)
weights = [edge[2]['weight'] for edge in edges]
log_weights = np.log(np.ones(1) + weights)

nx.draw(G, pos, with_labels=False, node_size=node_sizes, font_size=12, font_color='black', node_color='skyblue', edge_color='gray', width=log_weights, edge_cmap=plt.cm.Blues, alpha=0.8)

# Adjust labels to avoid overlap
texts = []
for node, (x, y) in pos.items():
    texts.append(plt.text(x, y, node, fontsize=12, ha='center', va='center'))

adjust_text(texts, arrowprops=dict(arrowstyle='-', color='black', lw=0.7))

plt.title('Network Visualization of Top 50 Topics')
plt.show()